# Menangani Data Tidak Seimbang

Empat skenario klasifikasi biner dipakai untuk menunjukkan masalah data tidak seimbang: deteksi penipuan kartu kredit, prediksi churn pelanggan, dan estimasi jenis tumor.

| Tugas | Rasio kelas |
| --- | --- |
| Deteksi penipuan kartu kredit | sangat timpang |
| Prediksi churn pelanggan | sekitar 4 banding 1 |
| Estimasi jenis tumor | hampir seimbang |

## Tujuan

- Mengenali pola data tidak seimbang.
- Menerapkan pembobotan kelas.
- Menerapkan resampling, yaitu oversampling dan undersampling.
- Membandingkan kinerja model dengan metrik yang tepat.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 123


## Fungsi bantu

Akurasi saja menyesatkan pada data timpang, karena model bisa menebak kelas mayoritas terus dan tetap terlihat akurat. Karena itu dipakai precision, recall, F-beta, dan ROC-AUC. Nilai beta 5 dipakai agar kesalahan pada kelas positif dihitung lima kali lebih mahal.


In [ ]:
def split_dataset(frame):
    features = frame.loc[:, frame.columns != "Class"]
    target = frame["Class"].astype("int")
    return train_test_split(
        features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
    )


def evaluate(actual, predicted, label="Original"):
    precision, recall, f_beta, _ = precision_recall_fscore_support(
        actual, predicted, beta=5, pos_label=1, average="binary", zero_division=0
    )
    return {
        "type": label,
        "accuracy": accuracy_score(actual, predicted),
        "recall": recall,
        "auc": roc_auc_score(actual, predicted),
        "precision": precision,
        "fscore": f_beta,
    }


def resample_training(features, target):
    positive = target[target == 1].index
    negative = target[target == 0].index

    oversampled_index = np.concatenate(
        [negative, np.random.choice(positive, size=len(negative), replace=True)]
    )
    undersampled_index = np.concatenate(
        [positive, np.random.choice(negative, size=len(positive), replace=False)]
    )
    return (
        features.loc[oversampled_index], target.loc[oversampled_index],
        features.loc[undersampled_index], target.loc[undersampled_index],
    )


def visualize_metrics(results):
    table = pd.DataFrame(results).set_index("type")
    print(table)
    table[["accuracy", "recall", "auc", "precision", "fscore"]].plot.bar(
        figsize=(12, 6)
    )
    plt.ylabel("Score")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## Skenario 1: Deteksi penipuan kartu kredit

Data transaksi kartu kredit dengan kelas penipuan yang sangat langka.


In [ ]:
CREDIT_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/im_credit.csv"
)
credit_data = pd.read_csv(CREDIT_URL, index_col=False)
credit_data.head()


In [ ]:
credit_data["Class"].value_counts()


In [ ]:
credit_data["Class"].value_counts().plot.bar(color=["green", "red"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


Transaksi penipuan hanya sebagian sangat kecil dari data, sehingga batang kelas positif hampir tidak terlihat.


In [ ]:
X_train, X_test, y_train, y_test = split_dataset(credit_data)

baseline_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)
evaluate(y_test, baseline_predictions)


Akurasi terlihat hampir sempurna, tetapi nilai itu menyesatkan: model hanya menebak kelas mayoritas. Precision, recall, dan F-beta menunjukkan kinerja sebenarnya.


## Oversampling dan undersampling

Oversampling menggandakan sampel kelas minoritas, sedangkan undersampling mengurangi sampel kelas mayoritas. Keduanya hanya diterapkan pada data latih agar data uji tetap mencerminkan sebaran asli.


In [ ]:
X_over, y_over, X_under, y_under = resample_training(X_train, y_train)
y_over.value_counts()


In [ ]:
over_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
over_model.fit(X_over, y_over)
over_predictions = over_model.predict(X_test)
evaluate(y_test, over_predictions, "Oversampling")


Oversampling menaikkan recall dan AUC, tetapi precision turun tajam karena data sintetis mendorong model terlalu agresif menandai penipuan.


## Pembobotan kelas

Pembobotan kelas mengubah fungsi kerugian tanpa menambah baris data. Bobot besar pada kelas minoritas membuat kesalahannya dihitung lebih mahal.


In [ ]:
weight_model = LogisticRegression(
    random_state=RANDOM_STATE, max_iter=1000, class_weight={0: 0.1, 1: 0.9}
)
weight_model.fit(X_train, y_train)
weight_predictions = weight_model.predict(X_test)
evaluate(y_test, weight_predictions, "Class Weight")


Pembobotan kelas memberi keseimbangan precision dan recall yang lebih baik daripada oversampling pada data yang sangat timpang.


## Skenario 2: Prediksi churn pelanggan

Data pelanggan telekomunikasi dengan rasio kelas sekitar empat banding satu.


In [ ]:
CHURN_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/im_churn.csv"
)
churn_data = pd.read_csv(CHURN_URL, index_col=False)
X_train, X_test, y_train, y_test = split_dataset(churn_data)
y_train.value_counts().plot.bar(color=["green", "red"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


In [ ]:
def build_forest(features, target, class_weight=None):
    model = RandomForestClassifier(
        n_estimators=50, max_depth=10, min_samples_split=5,
        class_weight=class_weight, random_state=RANDOM_STATE, n_jobs=-1,
    ).fit(features, target)
    return model.predict(X_test), model


In [ ]:
churn_results = []
predictions, _ = build_forest(X_train, y_train)
churn_results.append(evaluate(y_test, predictions, "Original"))

weighted_predictions, _ = build_forest(X_train, y_train, {0: 0.2, 1: 0.8})
churn_results.append(evaluate(y_test, weighted_predictions, "Class Weight"))


In [ ]:
X_over, y_over, X_under, y_under = resample_training(X_train, y_train)

predictions, _ = build_forest(X_over, y_over)
churn_results.append(evaluate(y_test, predictions, "Oversampling"))

predictions, _ = build_forest(X_under, y_under)
churn_results.append(evaluate(y_test, predictions, "Undersampling"))

visualize_metrics(churn_results)


Semua model tetap akurat, tetapi recall dan F-beta naik setelah pembobotan kelas atau resampling. Undersampling biasanya paling seimbang karena tidak menambah data palsu.


## Skenario 3: Estimasi jenis tumor

Data tumor yang hampir seimbang, dipakai sebagai pembanding bahwa resampling tidak selalu diperlukan.


In [ ]:
CANCER_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/im_cancer.csv"
)
cancer_data = pd.read_csv(CANCER_URL, index_col=False)
X_train, X_test, y_train, y_test = split_dataset(cancer_data)

X_over, y_over, X_under, y_under = resample_training(X_train, y_train)

cancer_results = []
predictions, _ = build_forest(X_train, y_train)
cancer_results.append(evaluate(y_test, predictions, "Original"))

predictions, _ = build_forest(X_train, y_train, {0: 0.2, 1: 0.8})
cancer_results.append(evaluate(y_test, predictions, "Class Weight"))

predictions, _ = build_forest(X_over, y_over)
cancer_results.append(evaluate(y_test, predictions, "Oversampling"))

predictions, _ = build_forest(X_under, y_under)
cancer_results.append(evaluate(y_test, predictions, "Undersampling"))

visualize_metrics(cancer_results)


Pada data yang hampir seimbang, model awal sudah baik. Resampling justru bisa menurunkan precision karena mengubah sebaran kelas secara buatan.


## Latihan: prediksi perubahan pekerjaan

Data HR yang timpang dipakai untuk menerapkan seluruh langkah sebelumnya.


In [ ]:
HR_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/im_hr.csv"
)
hr_data = pd.read_csv(HR_URL, index_col=False)
hr_data["Class"].value_counts().plot.bar(color=["green", "red"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = split_dataset(hr_data)
X_over, y_over, X_under, y_under = resample_training(X_train, y_train)

hr_results = []
predictions, _ = build_forest(X_train, y_train)
hr_results.append(evaluate(y_test, predictions, "Original"))

predictions, _ = build_forest(X_train, y_train, {0: 0.2, 1: 0.8})
hr_results.append(evaluate(y_test, predictions, "Class Weight"))

predictions, _ = build_forest(X_over, y_over)
hr_results.append(evaluate(y_test, predictions, "Oversampling"))

predictions, _ = build_forest(X_under, y_under)
hr_results.append(evaluate(y_test, predictions, "Undersampling"))

visualize_metrics(hr_results)


## Ringkasan

Akurasi tidak cukup untuk menilai model pada data timpang. Pembobotan kelas biasanya memberi perbaikan recall tanpa mengubah data, sedangkan resampling berguna bila kelas minoritas sangat sedikit. Pada data yang sudah seimbang, intervensi semacam ini justru bisa merugikan.
